<a href="https://colab.research.google.com/github/salza-pth/walmart-sales-forecasting/blob/main/Walmart_Sales_Forecasting_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Instalasi dan Inisialisasi PySpark

In [ ]:
!pip install findspark -q
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('WalmartSalesForecast').getOrCreate()
print('PySpark session sudah berhasil diinisialisasi')

## 2. Load Dataset

In [ ]:
import pandas as pd
from pyspark.sql.functions import col, to_date

# Load ke Pandas untuk proses EDA diawal
df_train    = pd.read_csv('/content/train - Walmart Sales Forecast.csv')
df_test     = pd.read_csv('/content/test - Walmart Sales Forecast.csv')
df_features = pd.read_csv('/content/features - Walmart Sales Forecast.csv')
df_stores   = pd.read_csv('/content/stores - Walmart Sales Forecast.csv')

# Load ke dalam PySpark
df_train_spark    = spark.read.csv('/content/train - Walmart Sales Forecast.csv',    header=True, inferSchema=True)
df_test_spark     = spark.read.csv('/content/test - Walmart Sales Forecast.csv',     header=True, inferSchema=True)
df_features_spark = spark.read.csv('/content/features - Walmart Sales Forecast.csv', header=True, inferSchema=True)
df_stores_spark   = spark.read.csv('/content/stores - Walmart Sales Forecast.csv',   header=True, inferSchema=True)

# Check jumlah kuantitas data sudah sesuai di PySpark
print('df_train head:')
print(df_train.head())
print('Jumlah baris train:', df_train_spark.count())

print('\ndf_test head:')
print(df_test.head())
print('Jumlah baris test: ', df_test_spark.count())

print('\ndf_features head:')
print(df_features.head())
print('Jumlah baris features:', df_features_spark.count())

print('\ndf_stores head:')
print(df_stores.head())
print('Jumlah baris stores:', df_stores_spark.count())

In [ ]:
# Check jenis tipe data setiap kolom
print('df_train_spark schema and head:')
df_train_spark.printSchema()
df_train_spark.show(5)

print('\ndf_test_spark schema and head:')
df_test_spark.printSchema()
df_test_spark.show(5)

print('\ndf_features_spark schema and head:')
df_features_spark.printSchema()
df_features_spark.show(5)

print('\ndf_stores_spark schema and head:')
df_stores_spark.printSchema()
df_stores_spark.show(5)

## 3. Eksplorasi Data Awal (EDA)

In [ ]:
# Info missing value, duplikasi dan statistik setiap dataset
print('df_Train_info')
df_train.info()
print(df_train.describe().T)
print('\nMissing values:')
print(df_train.isnull().sum())
print(df_train.duplicated().sum())

print('\ndf_Test_info ')
df_test.info()
print(df_test.describe().T)
print('\nMissing values:')
print(df_test.isnull().sum())
print(df_test.duplicated().sum())

print('\ndf_Features')
df_features.info()
print(df_features.describe().T)
print('\nMissing values:')
print(df_features.isnull().sum())
print(df_features.duplicated().sum())

print('\ndf_Stores')
df_stores.info()
print(df_stores.describe().T)
print('\nMissing values:')
print(df_stores.isnull().sum())
print(df_stores.duplicated().sum())

## 4. Penggabungan Dataset (Merge)

In [ ]:
from pyspark.sql.functions import col, to_date

# Konversi kolom Date
df_train_spark    = df_train_spark.withColumn('Date', to_date(col('Date'), 'yyyy-MM-dd'))
df_test_spark     = df_test_spark.withColumn('Date', to_date(col('Date'), 'yyyy-MM-dd'))
df_features_spark = df_features_spark.withColumn('Date', to_date(col('Date'), 'yyyy-MM-dd'))

# Menyamakan tipe IsHoliday ke boolean dulu sebelum join
df_train_spark    = df_train_spark.withColumn('IsHoliday', col('IsHoliday').cast('boolean'))
df_test_spark     = df_test_spark.withColumn('IsHoliday', col('IsHoliday').cast('boolean'))
df_features_spark = df_features_spark.withColumn('IsHoliday', col('IsHoliday').cast('boolean'))

# Menggabungkan train, features, dan stores
df_train_merged_spark = df_train_spark.join(df_features_spark, on=['Store', 'Date', 'IsHoliday'], how='left')
df_train_merged_spark = df_train_merged_spark.join(df_stores_spark, on=['Store'], how='left')

# Gabungkan test, features, dan stores
df_test_merged_spark = df_test_spark.join(df_features_spark, on=['Store', 'Date', 'IsHoliday'], how='left')
df_test_merged_spark = df_test_merged_spark.join(df_stores_spark, on=['Store'], how='left')

print('Train setelah merge:', df_train_merged_spark.count(), 'baris')
df_train_merged_spark.show(5)
print('Test setelah merge: ', df_test_merged_spark.count(), 'baris')
df_test_merged_spark.show(5)

In [ ]:
# Menggabungkan data di Pandas untuk mempermudah visualisasi
df_train_merged = pd.merge(df_train, df_features, on=['Store', 'Date', 'IsHoliday'], how='left')
df_train_merged = pd.merge(df_train_merged, df_stores, on=['Store'], how='left')
df_test_merged  = pd.merge(df_test,  df_features, on=['Store', 'Date', 'IsHoliday'], how='left')
df_test_merged  = pd.merge(df_test_merged,  df_stores, on=['Store'], how='left')

print('df_train_merged head:')
print(df_train_merged.head())
print('\n df_test_merged head:')
print(df_test_merged.head())


## 5. Data Cleaning

## 5a. Penanganan Missing Values dengan Pandas

In [ ]:
markdown_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']

#  Isi Markdown dengan 0
for col_name in markdown_cols:
    df_train_merged[col_name] = df_train_merged[col_name].fillna(0)
    df_test_merged[col_name] = df_test_merged[col_name].fillna(0)

# Isi CPI dan Unemployment dengan median
cpi_median = df_train_merged['CPI'].median()
unemployment_median = df_train_merged['Unemployment'].median()

df_train_merged['CPI'] = df_train_merged['CPI'].fillna(cpi_median)
df_train_merged['Unemployment'] = df_train_merged['Unemployment'].fillna(unemployment_median)
df_test_merged['CPI'] = df_test_merged['CPI'].fillna(cpi_median)
df_test_merged['Unemployment'] = df_test_merged['Unemployment'].fillna(unemployment_median)

print('df_train_merged head after handling missing values:')
print(df_train_merged.head())
print('\n--- df_train_merged Missing Values after imputation ---')
print(df_train_merged.isnull().sum())

## 5b. Penanganan Missing Values dengan PySpark


In [ ]:
from pyspark.sql.functions import col, lit, coalesce, expr, when

markdown_cols_and_others = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment']
markdown_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']

# Cast ke DOUBLE karena kolom terbaca string akibat nilai 'NA' di CSV
for col_name in markdown_cols_and_others:
    df_train_merged_spark = df_train_merged_spark.withColumn(
        col_name,
        expr(f"CASE WHEN upper(trim(CAST({col_name} AS STRING))) = 'NA' THEN CAST(NULL AS DOUBLE) ELSE try_cast(CAST({col_name} AS STRING) AS DOUBLE) END")
    )
    df_test_merged_spark = df_test_merged_spark.withColumn(
        col_name,
        expr(f"CASE WHEN upper(trim(CAST({col_name} AS STRING))) = 'NA' THEN CAST(NULL AS DOUBLE) ELSE try_cast(CAST({col_name} AS STRING) AS DOUBLE) END")
    )

for col_name in markdown_cols:
    df_train_merged_spark = df_train_merged_spark.withColumn(col_name, coalesce(col(col_name), lit(0.0)))
    df_test_merged_spark  = df_test_merged_spark.withColumn(col_name, coalesce(col(col_name), lit(0.0)))

# 3. Isi CPI dan Unemployment dengan median
cpi_median          = df_train_merged_spark.approxQuantile('CPI', [0.5], 0.01)[0]
unemployment_median = df_train_merged_spark.approxQuantile('Unemployment', [0.5], 0.01)[0]

df_train_merged_spark = df_train_merged_spark.withColumn('CPI', coalesce(col('CPI'), lit(cpi_median)))
df_train_merged_spark = df_train_merged_spark.withColumn('Unemployment', coalesce(col('Unemployment'), lit(unemployment_median)))
df_test_merged_spark  = df_test_merged_spark.withColumn('CPI', coalesce(col('CPI'), lit(cpi_median)))
df_test_merged_spark  = df_test_merged_spark.withColumn('Unemployment', coalesce(col('Unemployment'), lit(unemployment_median)))

print('df_train_merged_spark head after handling missing values:')
df_train_merged_spark.show(5)
print('\ndf_test_merged_spark head after handling missing values:')
df_test_merged_spark.show(5)

print('\n--- df_train_merged_spark Missing Values after imputation ---')
for col_name_check in markdown_cols_and_others:
    null_count = df_train_merged_spark.where(col(col_name_check).isNull()).count()
    if null_count > 0:
        print(f"Column '{col_name_check}' has {null_count} null values.")
df_train_merged_spark.summary().show()

print('\n--- df_test_merged_spark Missing Values after imputation ---')
for col_name_check in markdown_cols_and_others:
    null_count = df_test_merged_spark.where(col(col_name_check).isNull()).count()
    if null_count > 0:
        print(f"Column '{col_name_check}' has {null_count} null values.")
df_test_merged_spark.summary().show()

## 5c. Pengecekan dan Penghapusan Duplikasi

In [ ]:
print('--- Duplicates before dropping ---')
print(f"df_train_merged duplicate rows: {df_train_merged.duplicated().sum()}")
print(f"df_test_merged duplicate rows: {df_test_merged.duplicated().sum()}")

# Hapus duplikat
df_train_merged.drop_duplicates(inplace=True)
df_test_merged.drop_duplicates(inplace=True)

print('\n--- Duplicates after dropping ---')
print(f"df_train_merged duplicate rows: {df_train_merged.duplicated().sum()}")
print(f"df_test_merged duplicate rows: {df_test_merged.duplicated().sum()}")

print('\n--- DataFrame shapes after dropping duplicates ---')
print(f"df_train_merged shape: {df_train_merged.shape}")
print(f"df_test_merged shape: {df_test_merged.shape}")

In [ ]:
print('--- Duplicates before dropping ---')
print(f"df_train_merged_spark duplicate rows: {df_train_merged_spark.count() - df_train_merged_spark.dropDuplicates().count()}")
print(f"df_test_merged_spark duplicate rows: {df_test_merged_spark.count() - df_test_merged_spark.dropDuplicates().count()}")

# Hapus duplikat menggunakan PySpark
df_train_merged_spark = df_train_merged_spark.dropDuplicates()
df_test_merged_spark  = df_test_merged_spark.dropDuplicates()

print('\n--- Duplicates after dropping ---')
print(f"df_train_merged_spark duplicate rows: {df_train_merged_spark.count() - df_train_merged_spark.dropDuplicates().count()}")
print(f"df_test_merged_spark duplicate rows: {df_test_merged_spark.count() - df_test_merged_spark.dropDuplicates().count()}")

print('\n--- DataFrame shapes after dropping duplicates ---')
print(f"df_train_merged_spark shape: {df_train_merged_spark.count(), len(df_train_merged_spark.columns)}")
print(f"df_test_merged_spark shape: {df_test_merged_spark.count(), len(df_test_merged_spark.columns)}")

## 6. Transformasi Data

### 6a. Feature Engineering dari Kolom Date

In [ ]:
from pyspark.sql.functions import year, month, dayofmonth, weekofyear, dayofweek

# Ekstrak fitur waktu dari kolom Date (PySpark)
for df_spark in [df_train_merged_spark, df_test_merged_spark]:
    df_spark = df_spark

df_train_merged_spark = df_train_merged_spark.withColumn('Year', year(col('Date')))
df_train_merged_spark = df_train_merged_spark.withColumn('Month', month(col('Date')))
df_train_merged_spark = df_train_merged_spark.withColumn('Day', dayofmonth(col('Date')))
df_train_merged_spark = df_train_merged_spark.withColumn('Week', weekofyear(col('Date')))
df_train_merged_spark = df_train_merged_spark.withColumn('DayOfWeek', dayofweek(col('Date')) - 1)
df_train_merged_spark = df_train_merged_spark.withColumn('IsHoliday_int', col('IsHoliday').cast('int'))

df_test_merged_spark = df_test_merged_spark.withColumn('Year', year(col('Date')))
df_test_merged_spark = df_test_merged_spark.withColumn('Month', month(col('Date')))
df_test_merged_spark = df_test_merged_spark.withColumn('Day', dayofmonth(col('Date')))
df_test_merged_spark = df_test_merged_spark.withColumn('Week', weekofyear(col('Date')))
df_test_merged_spark = df_test_merged_spark.withColumn('DayOfWeek', dayofweek(col('Date')) - 1)
df_test_merged_spark = df_test_merged_spark.withColumn('IsHoliday_int', col('IsHoliday').cast('int'))

# Ekstrak fitur waktu dari kolom Date (Pandas)
for df_pd in [df_train_merged, df_test_merged]:
    df_pd['Date'] = pd.to_datetime(df_pd['Date'])
    df_pd['Year'] = df_pd['Date'].dt.year
    df_pd['Month'] = df_pd['Date'].dt.month
    df_pd['Day'] = df_pd['Date'].dt.day
    df_pd['Week'] = df_pd['Date'].dt.isocalendar().week.astype(int)
    df_pd['DayOfWeek'] = df_pd['Date'].dt.dayofweek
    df_pd['IsHoliday_int'] = df_pd['IsHoliday'].fillna(0).astype(int)

df_train_merged_spark.show(5)

### 6b. Aggregasi Data

In [ ]:
from pyspark.sql.functions import avg, sum as spark_sum

# Rata-rata penjualan mingguan keseluruhan
print('Rata-rata Weekly Sales:')
df_train_merged_spark.select(avg('Weekly_Sales')).show()

# Total penjualan per toko (top 10)
print('Top 10 Toko dengan Penjualan Tertinggi:')
df_train_merged_spark.groupBy('Store').agg(spark_sum('Weekly_Sales').alias('Total_Sales')).orderBy('Total_Sales', ascending=False).show(10)

## 7. Deteksi Outlier

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

numerical_cols = ['Weekly_Sales', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'Size']

# Box plot untuk deteksi outlier secara visual
plt.figure(figsize=(15, 10))
for i, col_name in enumerate(numerical_cols, 1):
    plt.subplot(2, 3, i)
    sns.boxplot(y=df_train_merged[col_name])
    plt.title(f'Box Plot of {col_name}')
    plt.ylabel(col_name)

plt.tight_layout()
plt.show()

# Ringkasan outlier dengan metode IQR
print('Ringkasan Outlier (Metode IQR):')
for col_name in numerical_cols:
    Q1 = df_train_merged[col_name].quantile(0.25)
    Q3 = df_train_merged[col_name].quantile(0.75)
    IQR = Q3 - Q1
    lb = Q1 - 1.5 * IQR
    ub = Q3 + 1.5 * IQR
    n_out = df_train_merged[(df_train_merged[col_name] < lb) | (df_train_merged[col_name] > ub)].shape[0]

    print(f'{col_name}: {n_out} outlier | batas bawah = {lb:.2f}, batas atas = {ub:.2f}')

## 8. Simpan Hasil Processed Dataset

In [ ]:
import os, shutil

output_path = '/content/walmart_processed.csv'

# Hapus folder lama dari run sebelumnya
if os.path.exists(output_path):
    os.remove(output_path)

# Simpan melalui pandas agar semua baris tersimpan dalam satu file
df_train_merged_spark.toPandas().to_csv(output_path, index=False)

print(f'Dataset tersimpan di: {output_path}')
print(f'Total baris tersimpan: {df_train_merged_spark.count()}')

## 9. Descriptive Analytics

In [ ]:
from pyspark.sql.functions import col, to_date

# Load kembali hasil processed dataset yang sudah bersih
df_ml = spark.read.csv('/content/walmart_processed.csv', header=True, inferSchema=True)

# Memastikan kolom Date bertipe date agar formatnya konsisten
df_ml = df_ml.withColumn('Date', to_date(col('Date'), 'yyyy-MM-dd'))

print('Ukuran Dataset untuk Machine Learning:')
print('Jumlah baris:', df_ml.count())
print('Jumlah kolom:', len(df_ml.columns))

print('\nStruktur Kolom (Schema):')
df_ml.printSchema()

print('\nRingkasan Statistik Deskriptif:')
df_ml.describe().show()

## 10. Visualisasi

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df_vis = df_train_merged.copy()

# Visualisasi 1: Tren penjualan mingguan
weekly = df_vis.groupby('Date')['Weekly_Sales'].mean().reset_index()
plt.figure(figsize=(14, 4))
plt.plot(weekly['Date'], weekly['Weekly_Sales'])
plt.title('Rata-rata Penjualan Mingguan')
plt.xlabel('Tanggal')
plt.ylabel('Weekly Sales')
plt.tight_layout()
plt.show()

# Visualisasi 2: Penjualan saat holiday vs non-holiday
plt.figure(figsize=(6, 4))
sns.boxplot(x='IsHoliday', y='Weekly_Sales', data=df_vis)
plt.title('Penjualan: Holiday vs Non-Holiday')
plt.tight_layout()
plt.show()

# Visualisasi 3: Total penjualan per tipe toko
plt.figure(figsize=(6, 4))
df_vis.groupby('Type')['Weekly_Sales'].mean().plot(kind='bar', color=['steelblue','orange','green'])
plt.title('Rata-rata Penjualan per Tipe Toko')
plt.xlabel('Tipe Toko')
plt.ylabel('Weekly Sales')
plt.tight_layout()
plt.show()

## 11. Menyimpan Hasil Data

In [ ]:
import os, shutil

# Simpan ke Parquet (storage layer / Data Lake)
output_parquet = '/content/walmart_processed'
if os.path.exists(output_parquet):
    shutil.rmtree(output_parquet)
df_train_merged_spark.write.parquet(output_parquet)
print(f'Parquet tersimpan di: {output_parquet}')

# Simpan ke CSV (untuk kebutuhan visualisasi Power BI)
output_csv = '/content/walmart_processed.csv'
if os.path.exists(output_csv):
    os.remove(output_csv)
df_train_merged_spark.toPandas().to_csv(output_csv, index=False)
print(f'CSV tersimpan di: {output_csv}')

print(f'Total baris tersimpan: {df_train_merged_spark.count()}')

## 12. Feature Engineering untuk Model

In [ ]:
# Daftar fitur yang dipakai model
feature_cols = [
    'Store', 'Dept',
    'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
    'Size', 'IsHoliday_int',
    'Year', 'Month', 'Week',
    'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5'
]

print('Jumlah fitur:', len(feature_cols))
print('Daftar fitur:', feature_cols)

## 13. Persiapan Data untuk Training

In [ ]:
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.sql.functions import col

# 1. Drop null
df_train_clean = df_train_merged_spark.dropna(subset=[
    'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
    'Size', 'IsHoliday_int', 'Year', 'Month', 'Week',
    'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5',
    'Weekly_Sales'
])

# 2. Encode Store
store_indexer_model = StringIndexer(inputCol="Store", outputCol="Store_index").fit(df_train_clean)
df_train_clean = store_indexer_model.transform(df_train_clean)

# 3. Encode Dept
dept_indexer_model = StringIndexer(inputCol="Dept",outputCol="Dept_index").fit(df_train_clean)
df_train_clean = dept_indexer_model.transform(df_train_clean)

# 4. One Hot Encoding
encoder_model = OneHotEncoder(inputCols=["Store_index", "Dept_index"],
    outputCols=["Store_vec", "Dept_vec"]).fit(df_train_clean)
df_train_clean = encoder_model.transform(df_train_clean)

# 5. Feature cols final
feature_cols_final = [
    'Store_vec', 'Dept_vec',
    'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
    'Size', 'IsHoliday_int',
    'Year', 'Month', 'Week',
    'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5'
]

# 6. Vector assembler
assembler = VectorAssembler(inputCols=feature_cols_final, outputCol='features')
data_train = assembler.transform(df_train_clean)

# 7. Split berdasarkan waktu
train_data = data_train.filter(col("Date") < "2012-06-01")
test_data  = data_train.filter(col("Date") >= "2012-06-01")

print(f"Training: {train_data.count()}")
print(f"Testing : {test_data.count()}")

In [ ]:
# 1. Menampilkan Jumlah Kolom Sebelum dan Sesudah Encoding / Feature Engineering
print("INFO STRUKTUR KOLOM DATASET: ")
print(f"Total Kolom Awal (Raw Data)            : 16 Kolom")
print(f"Total Kolom Setelah Preprocessing      : {len(df_train_clean.columns)} Kolom")
print(f"Nama Kolom Setelah Preprocessing       :")
print(df_train_clean.columns)

# 3. Menampilkan Perubahan Wujud DataFrame Sebelum vs Sesudah Vector Assembler
print("\nSTRUKTUR SEBELUM VECTOR ASSEMBLER: ")
# Mengambil sampel kolom fitur yang masih terpisah-pisah sebelum digabung
fitur_terpisah = [c for c in df_train_clean.columns if c not in ["Weekly_Sales", "Date"]]
df_train_clean.select(fitur_terpisah[:5]).show(5) # Menampilkan 5 kolom fitur pertama

print("\nSTRUKTUR SESUDAH VECTOR ASSEMBLER: ")
# Menampilkan kolom hasil training data yang sudah masuk Vector Assembler
train_data.select("features", "Weekly_Sales").show(5, truncate=False)

## 14. Model Regresi

## 14a. Linear Regression

In [ ]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql import functions as F
import math

# 1. Inisialisasi Model Linear Regression
lr = LinearRegression(featuresCol="features", labelCol="Weekly_Sales")

# 2. Fit the Model
lr_model = lr.fit(train_data)

# 3. Melakukan Prediksi
predictions = lr_model.transform(test_data)

# Melakukan Agregasi sebelum evaluasi
predictions_store = predictions.groupBy("Store", "Date").agg(
    F.sum("Weekly_Sales").alias("Actual_Store_Sales"),
    F.sum("prediction").alias("Pred_Store_Sales")
)

# 4. Tampilkan Hasil Perbandingan
predictions_store.select("Store", "Date", "Actual_Store_Sales", "Pred_Store_Sales").show(10)

# 5. Menghitung RMSE
error_df = predictions_store.withColumn(
    "squared_error",
    (F.col("Actual_Store_Sales") - F.col("Pred_Store_Sales")) ** 2
)
# Ambil rata-rata dari selisih kuadrat, lalu diakar kuadrat (Jadi RMSE Dolar)
mean_squared_error = error_df.select(F.mean("squared_error")).collect()[0][0]
rmse_dolar = math.sqrt(mean_squared_error)

# Ubah RMSE Dolar menjadi PERSENTASE dengan membaginya dengan Rata-rata Penjualan Aktual
avg_actual_sales = predictions_store.select(F.mean("Actual_Store_Sales")).collect()[0][0]
rmse_percentage = (rmse_dolar / avg_actual_sales) * 100

# 6. Evaluasi Metrics - R2 Score
r2_evaluator = RegressionEvaluator(labelCol="Actual_Store_Sales", predictionCol="Pred_Store_Sales", metricName="r2")
r2 = r2_evaluator.evaluate(predictions_store)

# 7. Laporan Performa
print("LAPORAN EVALUASI MODEL: LINEAR REGRESSION")
print(f"1. Akurasi Model (R² Score)        : {r2 * 100:.2f}%")
print(f"2. Nilai Error Model (RMSE)         : {rmse_percentage:.2f}%")
print(f"3. Estimasi Nilai Meleset (Dolar)   : ${rmse_dolar:,.2f} per minggu")

## 14b. Random Forest Regression

In [ ]:
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql import functions as F
import math

# 1. Inisialisasi Model
rf = RandomForestRegressor(featuresCol="features", labelCol="Weekly_Sales", numTrees=100, maxDepth=10, seed=42)

# 2. Training Model
rf_model = rf.fit(train_data)

# 3. Prediksi
rf_predictions = rf_model.transform(test_data)

# Melakukan Agregasi tiap toko sebelum evaluasi
rf_predictions_store = rf_predictions.groupBy("Store", "Date").agg(
    F.sum("Weekly_Sales").alias("Actual_Store_Sales"),
    F.sum("prediction").alias("Pred_Store_Sales")
)

# 4. Tampilkan Hasil Prediksi
rf_predictions_store.select("Store", "Date", "Actual_Store_Sales", "Pred_Store_Sales").show(10)

# 5. Menghitung RMSE
rf_error_df = rf_predictions_store.withColumn(
    "squared_error",
    (F.col("Actual_Store_Sales") - F.col("Pred_Store_Sales")) ** 2
)
rf_mean_squared_error = rf_error_df.select(F.mean("squared_error")).collect()[0][0]
rf_rmse_dolar = math.sqrt(rf_mean_squared_error)

# Ubah RMSE Dolar menjadi presentase dengan membaginya dengan Rata-rata Penjualan Aktual
rf_avg_actual_sales = rf_predictions_store.select(F.mean("Actual_Store_Sales")).collect()[0][0]
rf_rmse_percentage = (rf_rmse_dolar / rf_avg_actual_sales) * 100

# 6. Evaluasi R2 Score
rf_r2_evaluator = RegressionEvaluator(labelCol="Actual_Store_Sales", predictionCol="Pred_Store_Sales", metricName="r2")
rf_r2 = rf_r2_evaluator.evaluate(rf_predictions_store)

# 7. Laporan Performa
print("LAPORAN EVALUASI MODEL: RANDOM FOREST")
print(f"1. Akurasi Model (R² Score)        : {rf_r2 * 100:.2f}%")
print(f"2. Nilai Error Model (RMSE)         : {rf_rmse_percentage:.2f}%")
print(f"3. Estimasi Nilai Meleset (Dolar)   : ${rf_rmse_dolar:,.2f} per minggu")

## 15. Interpretasi Hasil

Model **Linear Regression** digunakan sebagai baseline. Karena pola penjualan Walmart bersifat non-linear (naik-turun ekstrem saat musim liburan), performa Linear Regression cenderung terbatas.

Model **Random Forest Regression** dengan 100 pohon dan kedalaman 10 memberikan hasil yang jauh lebih baik karena mampu menangkap pola non-linear pada data penjualan retail.

Semakin kecil nilai **RMSE**, semakin kecil rata-rata error prediksi model dalam satuan dollar. Semakin mendekati 1 nilai **R²**, semakin besar proporsi variasi penjualan yang berhasil dijelaskan model.